# **Install Dependencies

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.41.2

Found existing installation: transformers 5.8.1
Uninstalling transformers-5.8.1:
  Successfully uninstalled transformers-5.8.1
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2


In [ ]:
!pip install flask-ngrok flask-cors accelerate bitsandbytes sentencepiece

In [ ]:
!pip install pyngrok

# **Create the Server Script**

In [ ]:
# 1. Update the libraries
!pip install -q -U bitsandbytes transformers accelerate peft huggingface_hub pyngrok

from flask import Flask, request, jsonify
from flask_cors import CORS
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

# =========================
# 1. TURBO LOAD (Phi-3 native)
# =========================
model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("⏳ Loading model into GPU...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"
)

@app.route('/chat', methods=['POST'])
def chat():
    data = request.json
    user_message = data.get("message", "")
    history = data.get("history", []) # Now accepting history!
    system_prompt = data.get("system_prompt", "You are a professional interior designer.")
    
    # BUILD PHI-3 PROMPT TEMPLATE
    full_prompt = f"<|system|>\n{system_prompt}<|end|>\n"
    
    # Add history for context
    for turn in history:
        role = turn.get('role', 'user')
        content = turn.get('content', '')
        full_prompt += f"<|{role}|>\n{content}<|end|>\n"
    
    # Add current message
    full_prompt += f"<|user|>\n{user_message}<|end|>\n<|assistant|>\n"
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=300, # Longer responses!
            temperature=0.7, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # FIX: We don't skip special tokens here so we can find the assistant tag accurately
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # EXTRACT ONLY THE ASSISTANT REPLY
    try:
        # Phi-3 specific splitting
        final_reply = decoded.split("<|assistant|>")[-1].split("<|end|>")[0].strip()
    except:
        final_reply = decoded.split("<|assistant|>")[-1].strip()
    
    # Final cleanup of any remaining special tokens
    final_reply = final_reply.replace("<|end|>", "").replace("<|assistant|>", "").strip()
    
    return jsonify({"response": final_reply})

# =========================
# 2. START NGROK (Use your Token)
# =========================
NGROK_TOKEN = "3DfWEppGHvKHw1COuMZDQUMlgB1_7hJEiPeVNLk8xefL9KzJb"
ngrok.set_auth_token(NGROK_TOKEN)
try:
    [ngrok.disconnect(t.public_url) for t in ngrok.get_tunnels()]
except: pass

public_url = ngrok.connect(5000)
print(f"🚀 STABLE URL: {public_url}")

if __name__ == "__main__":
    app.run(port=5000)


⏳ Loading model into GPU...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

🚀 STABLE URL: NgrokTunnel: "https://transfer-certainty-wick.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [15/May/2026 08:46:34] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:13] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:13] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:17] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:18] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:20] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:23] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:27] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:30] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:40] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 08:48:44] "POST /chat HTTP/1.1" 200 -
